In [1]:
import os
import shutil
import shlex
import subprocess
import demucs.separate
from df.enhance import enhance, init_df, load_audio, save_audio
import json
import requests
import time

def transcribe_with_whisperx(audio_file_path, language=None, enable_diarization=True, min_speakers=1, max_speakers=10, whisperx_url="http://whisperx:5001"):
    """Transcribe audio using WhisperX API service with diarization support."""
    print(f"Transcribing audio file with WhisperX: {audio_file_path}")
    
    # Check if WhisperX service is available
    try:
        health_response = requests.get(f"{whisperx_url}/health", timeout=10)
        if health_response.status_code != 200:
            raise Exception("WhisperX service not healthy")
        print("✅ WhisperX service is healthy")
    except Exception as e:
        print(f"❌ WhisperX service unavailable: {e}")
        return None
    
    # Prepare the request
    with open(audio_file_path, 'rb') as audio_file:
        files = {'audio': audio_file}
        data = {
            'language': language or 'en',
            'diarize': str(enable_diarization).lower(),
            'min_speakers': str(min_speakers),
            'max_speakers': str(max_speakers),
            'format': 'json'
        }
        
        print(f"📤 Sending audio to WhisperX API...")
        print(f"   Language: {data['language']}")
        print(f"   Diarization: {data['diarize']}")
        print(f"   Speaker range: {min_speakers}-{max_speakers}")
        
        try:
            response = requests.post(
                f"{whisperx_url}/transcribe",
                files=files,
                data=data,
                timeout=300  # 5 minutes timeout
            )
            
            if response.status_code == 200:
                result = response.json()
                print(f"✅ Transcription complete. Found {len(result.get('segments', []))} segments.")
                
                # Print preview with speaker information if available
                for i, segment in enumerate(result.get('segments', [])[:5]):  # First 5 segments
                    start = segment.get('start', 0)
                    end = segment.get('end', 0)
                    text = segment.get('text', '').strip()
                    speaker = segment.get('speaker', None)
                    
                    if speaker:
                        print(f"[{start:06.1f} --> {end:06.1f}] [{speaker}] {text}")
                    else:
                        print(f"[{start:06.1f} --> {end:06.1f}] {text}")
                
                if len(result.get('segments', [])) > 5:
                    print("...")
                
                return result
            else:
                print(f"❌ WhisperX API error: {response.status_code}")
                print(f"Response: {response.text}")
                return None
                
        except requests.exceptions.Timeout:
            print("❌ WhisperX API timeout - file might be too large")
            return None
        except Exception as e:
            print(f"❌ Error calling WhisperX API: {e}")
            return None

def separate_vocals(input_audio_path, output_dir, model='mdx_extra'):
    """
    Separate vocals and no-vocals with Demucs and save to custom output_dir.
    """
    print(f"Separating vocals from {input_audio_path}...")

    # Run Demucs CLI separation
    cmd = f'--two-stems vocals -n {model} "{input_audio_path}"'
    demucs.separate.main(shlex.split(cmd))

    base_name = os.path.splitext(os.path.basename(input_audio_path))[0]
    demucs_output_dir = os.path.join("separated", model, base_name)

    vocals_src = os.path.join(demucs_output_dir, "vocals.wav")
    no_vocals_src = os.path.join(demucs_output_dir, "no_vocals.wav")

    os.makedirs(output_dir, exist_ok=True)

    vocals_dst = os.path.join(output_dir, f"{base_name}_vocals.wav")
    no_vocals_dst = os.path.join(output_dir, f"{base_name}_no_vocals.wav")

    shutil.copy2(vocals_src, vocals_dst)
    shutil.copy2(no_vocals_src, no_vocals_dst)

    print(f"Vocals saved to: {vocals_dst}")
    print(f"No vocals saved to: {no_vocals_dst}")

    return vocals_dst, no_vocals_dst  # Return paths of saved files

def convert_to_48k_wav(input_audio_path, output_wav_path):
    """
    Convert any audio input to 48kHz stereo WAV using ffmpeg.
    """
    print(f"Converting {input_audio_path} to 48kHz WAV at {output_wav_path}...")
    subprocess.run([
        'ffmpeg', '-y', '-i', input_audio_path,
        '-ar', '48000',
        '-ac', '2',
        output_wav_path
    ], check=True)
    print("Conversion done.")

def denoise_audio(input_wav_path, output_wav_path):
    """
    Denoise/enhance the WAV audio using DeepFilterNet.
    """
    print(f"Denoising audio {input_wav_path} to {output_wav_path}...")
    model, df_state, _ = init_df()
    audio, _ = load_audio(input_wav_path, sr=df_state.sr())
    enhanced = enhance(model, df_state, audio)
    save_audio(output_wav_path, enhanced, df_state.sr())
    print("Denoising done.")

def separate_audio_from_video(video_path, audio_out_path, muted_video_out_path):
    """Extract audio from video and save both audio and muted video separately."""
    
    # Ensure output directories exist
    os.makedirs(os.path.dirname(audio_out_path), exist_ok=True)
    os.makedirs(os.path.dirname(muted_video_out_path), exist_ok=True)
    
    # Extract audio (48 kHz mono WAV)
    audio_cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le", "-ar", "48000", "-ac", "2",
        audio_out_path
    ]

    # Remove audio from video
    muted_video_cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-an", muted_video_out_path
    ]

    print("🔊 Extracting audio...")
    subprocess.run(audio_cmd, check=True)
    print(f"✅ Audio saved to: {audio_out_path}")

    print("🎥 Saving muted video...")
    subprocess.run(muted_video_cmd, check=True)
    print(f"✅ Muted video saved to: {muted_video_out_path}")

def write_transcription_to_json(result, output_path):
    """Write the WhisperX transcription result to a JSON file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)  # Create output dir
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"✅ JSON transcript saved to: {output_path}")

def write_transcription_to_srt(result, output_path):
    """Convert WhisperX result to SRT format and save."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Debug: Print the structure of result
    print("🔍 Debug - Result keys:", list(result.keys()) if isinstance(result, dict) else "Not a dict")
    print("🔍 Debug - Segments count:", len(result.get('segments', [])) if isinstance(result, dict) else "No segments key")
    
    # If segments exist, print the first one
    segments = result.get('segments', []) if isinstance(result, dict) else []
    if segments:
        print("🔍 Debug - First segment:", segments[0])
    else:
        print("🔍 Debug - No segments found!")
        print("🔍 Debug - Full result:", result)
    
    srt_content = ""
    for i, segment in enumerate(segments, 1):
        start_time = format_timestamp_srt(segment.get('start', 0))
        end_time = format_timestamp_srt(segment.get('end', 0))
        text = segment.get('text', '').strip()
        speaker = segment.get('speaker', '')
        
        print(f"🔍 Debug - Segment {i}: start={start_time}, end={end_time}, text='{text}', speaker='{speaker}'")
        
        if speaker:
            text = f"[{speaker}] {text}"
        
        srt_content += f"{i}\n{start_time} --> {end_time}\n{text}\n\n"
    
    print(f"🔍 Debug - SRT content length: {len(srt_content)} characters")
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(srt_content)
    
    print(f"✅ SRT transcript saved to: {output_path}")


def format_timestamp_srt(seconds):
    """Convert seconds to SRT timestamp format (HH:MM:SS,mmm)."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    milliseconds = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}"



/venv/lib/python3.10/site-packages/df/io.py:9: UserWarning: `torchaudio.backend.common.AudioMetaData` has been moved to `torchaudio.AudioMetaData`. Please update the import path.
  from torchaudio.backend.common import AudioMetaData


In [2]:
def generate_pipeline_paths(video_path, base_output="../output"):
    video_title = os.path.splitext(os.path.basename(video_path))[0]
    root_dir = os.path.join(base_output, video_title)

    return {
        "root": root_dir,
        "audio_raw": os.path.join(root_dir, "audio", "original_unprocessed.wav"),
        "audio_vocals": os.path.join(root_dir, "audio", "vocals_raw.wav"),
        "audio_vocals_48k": os.path.join(root_dir, "audio", "vocals_48k.wav"),
        "audio_vocals_denoised": os.path.join(root_dir, "audio", "vocals_denoised.wav"),
        "video_muted": os.path.join(root_dir, "video", "video_muted.mp4"),
        "video_final": os.path.join(root_dir, "video", "video_final.mp4"),
        "transcript_json": os.path.join(root_dir, "transcripts", "transcript.json"),
        "demucs_out": os.path.join(root_dir, "demucs")
    }


In [3]:
def full_video_audio_pipeline(video_path, base_output="../output", enable_diarization=True, min_speakers=1, max_speakers=10, language=None):
    """
    Complete pipeline: video → audio separation → vocal isolation → denoising → WhisperX transcription with diarization
    Skips preprocessing steps if intermediate files already exist.
    """
    from df.enhance import enhance, init_df, load_audio, save_audio

    paths = generate_pipeline_paths(video_path, base_output)
    os.makedirs(paths["root"], exist_ok=True)

    # Step 1: Separate audio and muted video (skip if files exist)
    if not os.path.exists(paths["audio_raw"]) or not os.path.exists(paths["video_muted"]):
        print("🎵 Separating audio from video...")
        separate_audio_from_video(video_path, paths["audio_raw"], paths["video_muted"])
    else:
        print("✅ Raw audio and muted video already exist, skipping separation...")

    # Step 2: Separate vocals with Demucs (skip if vocals exist)
    if not os.path.exists(paths["audio_vocals"]):
        print("🎤 Separating vocals with Demucs...")
        vocals_path, _ = separate_vocals(paths["audio_raw"], output_dir=os.path.dirname(paths["audio_vocals"]))
        shutil.move(vocals_path, paths["audio_vocals"])  # Rename into our structure
    else:
        print("✅ Vocals already separated, skipping Demucs...")

    # Step 3: Convert vocals to 48kHz WAV (skip if converted file exists)
    if not os.path.exists(paths["audio_vocals_48k"]):
        print("🔄 Converting vocals to 48kHz WAV...")
        convert_to_48k_wav(paths["audio_vocals"], paths["audio_vocals_48k"])
    else:
        print("✅ 48kHz vocals already exist, skipping conversion...")

    # Step 4: Denoise vocals (skip if denoised file exists)
    if not os.path.exists(paths["audio_vocals_denoised"]):
        print("🔧 Denoising vocals...")
        model_df, df_state, _ = init_df()
        audio, _ = load_audio(paths["audio_vocals_48k"], sr=df_state.sr())
        enhanced = enhance(model_df, df_state, audio)
        save_audio(paths["audio_vocals_denoised"], enhanced, df_state.sr())
    else:
        print("✅ Denoised vocals already exist, skipping denoising...")

    # Step 5: Transcribe with WhisperX API (always run unless transcript exists)
    # Generate transcript paths if not in the paths dict
    transcript_json_path = paths.get("transcript_json", os.path.join(paths["root"], "transcript.json"))
    transcript_srt_path = paths.get("transcript_srt", os.path.join(paths["root"], "transcript.srt"))
    
    if not os.path.exists(transcript_json_path) or not os.path.exists(transcript_srt_path):
        print("🎤 Starting transcription with WhisperX...")
        result = transcribe_with_whisperx(
            paths["audio_raw"], 
            language=language,
            enable_diarization=enable_diarization,
            min_speakers=min_speakers,
            max_speakers=max_speakers
        )
        
        if result:
            write_transcription_to_json(result, transcript_json_path)
        else:
            print("❌ Transcription failed!")
    else:
        print("✅ Transcription files already exist, skipping WhisperX...")

    print(f"🎉 Pipeline complete. All outputs saved under: {paths['root']}")
    return paths


In [4]:
import json
import os
import subprocess
from typing import Dict, List, Tuple
from collections import defaultdict

def separate_audio_by_speaker(transcript_json_path: str, original_audio_path: str, output_dir: str) -> Dict[str, str]:
    """
    Separate audio file into individual speaker segments based on WhisperX transcript.
    
    Args:
        transcript_json_path: Path to the JSON transcript file
        original_audio_path: Path to the original audio file
        output_dir: Directory to save separated audio files
    
    Returns:
        Dictionary mapping speaker IDs to their audio file paths
    """
    
    # Load transcript
    with open(transcript_json_path, 'r', encoding='utf-8') as f:
        transcript_data = json.load(f)
    
    # Extract segments from the nested structure
    segments = transcript_data.get('result', {}).get('segments', [])
    debug_info = transcript_data.get('debug_info', {})
    
    print(f"📊 Loaded transcript with {debug_info.get('segments_count', len(segments))} segments")
    print(f"🎤 Found {debug_info.get('speakers_count')} speakers")
    print(f"⏱️  Audio duration: {debug_info.get('audio_duration', 'unknown')} seconds")
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Group segments by speaker
    speaker_segments = defaultdict(list)
    
    for segment in segments:
        speaker = segment.get('speaker')
        if not speaker:
            continue
            
        speaker_segments[speaker].append({
            'start': segment.get('start', 0),
            'end': segment.get('end', 0),
            'text': segment.get('text', '').strip()
        })
    
    print(f"🎤 Processing speakers: {list(speaker_segments.keys())}")
    
    # Print speaker statistics
    for speaker, segs in speaker_segments.items():
        total_duration = sum(seg['end'] - seg['start'] for seg in segs)
        print(f"   {speaker}: {len(segs)} segments, {total_duration:.1f}s total")
    
    # Create audio files for each speaker
    speaker_files = {}
    
    for speaker_id, segments in speaker_segments.items():
        print(f"\n🔄 Processing {speaker_id} ({len(segments)} segments)...")
        
        # Create filter complex for ffmpeg
        filter_parts = []
        
        for i, segment in enumerate(segments):
            start_time = segment['start']
            duration = segment['end'] - segment['start']
            
            # Skip very short segments (less than 0.1 seconds)
            if duration < 0.1:
                continue
            
            # Add filter for this segment
            filter_parts.append(f"[0:a]atrim=start={start_time}:duration={duration},asetpts=PTS-STARTPTS[a{i}]")
        
        if not filter_parts:
            print(f"   ⚠️ No valid segments found for {speaker_id}")
            continue
            
        # Combine all segments for this speaker
        if len(filter_parts) == 1:
            filter_complex = filter_parts[0].replace(f"[a0]", "[out]")
        else:
            filter_complex = ";".join(filter_parts)
            # Concatenate all segments
            inputs_list = "".join([f"[a{i}]" for i in range(len(filter_parts))])
            filter_complex += f";{inputs_list}concat=n={len(filter_parts)}:v=0:a=1[out]"
        
        # Output file path
        speaker_file = os.path.join(output_dir, f"{speaker_id}.wav")
        speaker_files[speaker_id] = speaker_file
        
        # FFmpeg command
        cmd = [
            'ffmpeg', '-y',
            '-i', original_audio_path,
            '-filter_complex', filter_complex,
            '-map', '[out]',
            '-c:a', 'pcm_s16le',
            '-ar', '48000',
            speaker_file
        ]
        
        try:
            print(f"   Extracting audio for {speaker_id}...")
            subprocess.run(cmd, check=True, capture_output=True, text=True)
            print(f"   ✅ Saved: {speaker_file}")
        except subprocess.CalledProcessError as e:
            print(f"   ❌ Error processing {speaker_id}: {e}")
            print(f"   Command: {' '.join(cmd)}")
            continue
    
    return speaker_files

def create_speaker_transcript_files(transcript_json_path: str, output_dir: str) -> Dict[str, str]:
    """
    Create individual transcript files for each speaker.
    
    Args:
        transcript_json_path: Path to the JSON transcript file
        output_dir: Directory to save speaker transcript files
    
    Returns:
        Dictionary mapping speaker IDs to their transcript file paths
    """
    
    # Load transcript
    with open(transcript_json_path, 'r', encoding='utf-8') as f:
        transcript_data = json.load(f)
    
    segments = transcript_data.get('result', {}).get('segments', [])
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Group segments by speaker
    speaker_segments = defaultdict(list)
    
    for segment in segments:
        speaker = segment.get('speaker')
        if not speaker:
            continue
            
        speaker_segments[speaker].append({
            'start': segment.get('start', 0),
            'end': segment.get('end', 0),
            'text': segment.get('text', '').strip(),
            'words': segment.get('words', [])
        })
    
    speaker_transcript_files = {}
    
    for speaker_id, segments in speaker_segments.items():
        # Create text transcript
        txt_file = os.path.join(output_dir, f"{speaker_id}_transcript.txt")
        srt_file = os.path.join(output_dir, f"{speaker_id}_transcript.srt")
        json_file = os.path.join(output_dir, f"{speaker_id}_transcript.json")
        
        # Write plain text transcript
        with open(txt_file, 'w', encoding='utf-8') as f:
            f.write(f"Speaker: {speaker_id}\n")
            f.write("=" * 50 + "\n\n")
            
            for segment in segments:
                timestamp = f"[{segment['start']:.1f}s - {segment['end']:.1f}s]"
                f.write(f"{timestamp} {segment['text']}\n")
        
        # Write SRT format
        with open(srt_file, 'w', encoding='utf-8') as f:
            for i, segment in enumerate(segments, 1):
                start_time = format_timestamp_srt(segment['start'])
                end_time = format_timestamp_srt(segment['end'])
                f.write(f"{i}\n")
                f.write(f"{start_time} --> {end_time}\n")
                f.write(f"{segment['text']}\n\n")
        
        # Write JSON format with detailed info
        with open(json_file, 'w', encoding='utf-8') as f:
            speaker_data = {
                'speaker_id': speaker_id,
                'total_segments': len(segments),
                'total_duration': sum(seg['end'] - seg['start'] for seg in segments),
                'segments': segments
            }
            json.dump(speaker_data, f, indent=2, ensure_ascii=False)
        
        speaker_transcript_files[speaker_id] = {
            'txt': txt_file,
            'srt': srt_file,
            'json': json_file
        }
        
        print(f"📝 Created transcripts for {speaker_id}:")
        print(f"   Text: {txt_file}")
        print(f"   SRT: {srt_file}")
        print(f"   JSON: {json_file}")
    
    return speaker_transcript_files

def format_timestamp_srt(seconds):
    """Convert seconds to SRT timestamp format (HH:MM:SS,mmm)."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    milliseconds = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}"

def create_speaker_separation_pipeline(base_output_dir: str, video_title: str):
    """
    Complete speaker separation pipeline using existing pipeline outputs.
    
    Args:
        base_output_dir: Base output directory (e.g., "../output")
        video_title: Video title (folder name)
    """
    
    root_dir = os.path.join(base_output_dir, video_title)
    
    # Paths
    transcript_json = os.path.join(root_dir, "transcripts", "transcript.json")
    original_audio = os.path.join(root_dir, "audio", "original_unprocessed.wav")
    
    # Output directories
    audio_output_dir = os.path.join(root_dir, "separated_audio")
    transcript_output_dir = os.path.join(root_dir, "separated_transcripts")
    
    print(f"🎬 Starting speaker separation for: {video_title}")
    print(f"📄 Transcript: {transcript_json}")
    print(f"🔊 Audio: {original_audio}")
    
    # Check if required files exist
    if not os.path.exists(transcript_json):
        print("❌ Transcript JSON not found. Run the main pipeline first.")
        return None
    
    if not os.path.exists(original_audio):
        print("❌ Original audio not found. Run the main pipeline first.")
        return None
    
    # Load and display transcript info
    with open(transcript_json, 'r', encoding='utf-8') as f:
        transcript_data = json.load(f)
    
    debug_info = transcript_data.get('debug_info', {})
    print(f"\n📊 Transcript Information:")
    print(f"   Total duration: {debug_info.get('audio_duration', 'unknown')} seconds")
    print(f"   Total segments: {debug_info.get('segments_count', 'unknown')}")
    print(f"   Number of speakers: {debug_info.get('speakers_count', 'unknown')}")
    print(f"   Text preview: {debug_info.get('text_preview', '')[:100]}...")
    
    # Create speaker transcript files
    print("\n📝 Creating individual speaker transcripts...")
    speaker_transcript_files = create_speaker_transcript_files(
        transcript_json, transcript_output_dir
    )
    
    # Separate audio by speaker
    print("\n🔊 Separating audio by speaker...")
    speaker_audio_files = separate_audio_by_speaker(
        transcript_json, original_audio, audio_output_dir
    )
    
    # Summary
    print(f"\n🎉 Speaker separation complete!")
    print(f"📁 Audio files saved to: {audio_output_dir}")
    print(f"📁 Transcript files saved to: {transcript_output_dir}")
    
    print("\n📊 Summary:")
    for speaker in speaker_audio_files.keys():
        print(f"   {speaker}: Audio ✅ Transcripts ✅")
    
    return {
        'audio_files': speaker_audio_files,
        'transcript_files': speaker_transcript_files,
        'output_dirs': {
            'audio': audio_output_dir,
            'transcripts': transcript_output_dir
        }
    }

# Usage example:
# Separate speakers from your processed video
video_title = "job"  # The name of your video (folder name)
base_output = "../output"

separation_results = create_speaker_separation_pipeline(base_output, video_title)


🎬 Starting speaker separation for: job
📄 Transcript: ../output/job/transcripts/transcript.json
🔊 Audio: ../output/job/audio/original_unprocessed.wav

📊 Transcript Information:
   Total duration: 820.9879375 seconds
   Total segments: 151
   Number of speakers: 3
   Text preview: Hi, Chris Sauer. Hi, Kate Johnson. Nice to meet you. Nice to meet you, Kate. Chris, please tell me a...

📝 Creating individual speaker transcripts...
📝 Created transcripts for SPEAKER_01:
   Text: ../output/job/separated_transcripts/SPEAKER_01_transcript.txt
   SRT: ../output/job/separated_transcripts/SPEAKER_01_transcript.srt
   JSON: ../output/job/separated_transcripts/SPEAKER_01_transcript.json
📝 Created transcripts for SPEAKER_00:
   Text: ../output/job/separated_transcripts/SPEAKER_00_transcript.txt
   SRT: ../output/job/separated_transcripts/SPEAKER_00_transcript.srt
   JSON: ../output/job/separated_transcripts/SPEAKER_00_transcript.json

🔊 Separating audio by speaker...
📊 Loaded transcript with 151 segme

In [5]:
class AudioTranscriber:
    def __init__(self, whisper_model_size="base", sensevoice_model_dir="iic/SenseVoiceSmall"):
        """Initialize both Whisper and SenseVoice models"""
        self.whisper_model = self.load_whisper_model(whisper_model_size)
        self.sensevoice_model = self.load_sensevoice_model(sensevoice_model_dir)
        
    def load_whisper_model(self, model_size="base"):
        """Load the Whisper model."""
        print(f"Loading Whisper {model_size} model...")
        model = whisper.load_model(model_size)
        print(f"Whisper {model_size} model loaded successfully.")
        return model

    def load_sensevoice_model(self, model_dir):
        """Load the SenseVoice model."""
        print("Loading SenseVoice model...")
        model = AutoModel(
            model=model_dir,
            trust_remote_code=True,
            remote_code="./model.py",    
            vad_model="fsmn-vad",
            vad_kwargs={"max_single_segment_time": 30000},
            device="cuda:0",
            disable_update=True,
        )
        print("SenseVoice model loaded successfully.")
        return model

    def transcribe_with_whisper(self, audio_file_path, language=None):
        """Transcribe audio using Whisper with detailed segments."""
        print(f"Transcribing audio file with Whisper: {audio_file_path}")
        
        options = {"word_timestamps": True}
        if language:
            options["language"] = language
        
        result = self.whisper_model.transcribe(audio_file_path, **options)
        print("Whisper transcription complete.")
        return result

    def extract_emotions_with_sensevoice(self, audio_file_path):
        """Extract emotions and transcription using SenseVoice."""
        print("Processing audio with SenseVoice for emotion detection...")
        
        if not os.path.exists(audio_file_path):
            raise FileNotFoundError(f"Input audio file not found: {audio_file_path}")

        try:
            res = self.sensevoice_model.generate(
                input=audio_file_path,
                cache={},
                language="auto",
                use_itn=True,
                batch_size_s=60,
                merge_vad=True,
                merge_length_s=15,
            )
        except Exception as e:
            print(f"Error during SenseVoice processing: {e}")
            raise

        text = rich_transcription_postprocess(res[0]["text"])
        print("SenseVoice processing complete.")
        return text

    def replace_emojis_with_tags(self, text):
        """Replace emojis with emotion tags."""
        emoji_to_tag = {
            "😊": "happy",
            "😡": "angry", 
            "😔": "sad",
            "😀": "laughter",
            "🎼": "music",
            "👏": "applause",
            "🤧": "cough & sneeze",
            "😭": "cry",
        }

        for emoji, tag in emoji_to_tag.items():
            text = text.replace(emoji, f"[{tag}]")

        return text.strip()

    def extract_emotions_from_text(self, text):
        """Extract emotion tags and their positions from text."""
        emotion_pattern = r'\[([^\]]+)\]'
        emotions = []
        
        for match in re.finditer(emotion_pattern, text):
            emotions.append({
                'emotion': match.group(1),
                'start_pos': match.start(),
                'end_pos': match.end(),
                'text_before': text[:match.start()],
                'text_after': text[match.end():]
            })
        
        # Remove emotion tags to get clean text for matching
        clean_text = re.sub(emotion_pattern, '', text).strip()
        clean_text = ' '.join(clean_text.split())  # Normalize whitespace
        
        return emotions, clean_text

    def fuzzy_match_segments(self, whisper_segments, sensevoice_text, emotions, threshold=0.6):
        """Match Whisper segments with SenseVoice emotions using fuzzy matching."""
        _, clean_sensevoice_text = self.extract_emotions_from_text(sensevoice_text)
        
        enhanced_segments = []
        
        for segment in whisper_segments:
            segment_text = segment['text'].strip()
            segment_copy = segment.copy()
            segment_copy['emotions'] = []
            
            # Find the best match in the SenseVoice text
            best_match_ratio = 0
            best_match_emotions = []
            
            # Try to find fuzzy matches for this segment
            for emotion in emotions:
                # Get text around the emotion for context
                emotion_context_start = max(0, emotion['start_pos'] - 50)
                emotion_context_end = min(len(sensevoice_text), emotion['end_pos'] + 50)
                emotion_context = sensevoice_text[emotion_context_start:emotion_context_end]
                
                # Remove emotion tags from context for comparison
                clean_context = re.sub(r'\[[^\]]+\]', '', emotion_context).strip()
                clean_context = ' '.join(clean_context.split())
                
                # Calculate similarity
                similarity = SequenceMatcher(None, segment_text.lower(), clean_context.lower()).ratio()
                
                if similarity > threshold and similarity > best_match_ratio:
                    best_match_ratio = similarity
                    best_match_emotions = [emotion['emotion']]
                elif similarity > threshold and abs(similarity - best_match_ratio) < 0.1:
                    # If similarity is close to best, add this emotion too
                    if emotion['emotion'] not in best_match_emotions:
                        best_match_emotions.append(emotion['emotion'])
            
            segment_copy['emotions'] = best_match_emotions
            segment_copy['emotion_confidence'] = best_match_ratio
            enhanced_segments.append(segment_copy)
        
        return enhanced_segments

    def process_audio(self, audio_file_path, output_dir, language=None):
        """Process audio with both models and create enhanced transcription."""
        
        # Ensure output directory exists
        os.makedirs(output_dir, exist_ok=True)
        
        # Get Whisper transcription
        whisper_result = self.transcribe_with_whisper(audio_file_path, language)
        
        # Get SenseVoice emotions
        sensevoice_text = self.extract_emotions_with_sensevoice(audio_file_path)
        sensevoice_tagged = self.replace_emojis_with_tags(sensevoice_text)
        
        # Extract emotions and clean text
        emotions, clean_sensevoice_text = self.extract_emotions_from_text(sensevoice_tagged)
        
        # Match segments with emotions
        enhanced_segments = self.fuzzy_match_segments(
            whisper_result['segments'], 
            sensevoice_tagged, 
            emotions
        )
        
        # Create enhanced result
        enhanced_result = {
            'text': whisper_result['text'],
            'language': whisper_result.get('language', 'unknown'),
            'segments': enhanced_segments,
            'sensevoice_raw': sensevoice_text,
            'sensevoice_tagged': sensevoice_tagged,
            'detected_emotions': [emotion['emotion'] for emotion in emotions]
        }
        
        # Save results
        self.save_results(enhanced_result, output_dir, audio_file_path)
        
        return enhanced_result

    def save_results(self, enhanced_result, output_dir, audio_file_path):
        """Save all results to files."""
        filename_base = os.path.splitext(os.path.basename(audio_file_path))[0]
        
        # Save enhanced JSON with emotions
        json_path = os.path.join(output_dir, f"{filename_base}_enhanced.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(enhanced_result, f, indent=2, ensure_ascii=False)
        print(f"✅ Enhanced JSON transcript saved to: {json_path}")
        
        # Save readable transcript with emotions
        txt_path = os.path.join(output_dir, f"{filename_base}_with_emotions.txt")
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write("TRANSCRIPT WITH EMOTIONS\n")
            f.write("="*50 + "\n\n")
            
            for segment in enhanced_result['segments']:
                timestamp = f"[{segment['start']:.1f}s - {segment['end']:.1f}s]"
                emotions_str = f" {segment['emotions']}" if segment['emotions'] else ""
                f.write(f"{timestamp}{emotions_str}: {segment['text']}\n")
            
            f.write(f"\n\nDetected Emotions: {', '.join(enhanced_result['detected_emotions'])}")
        
        print(f"✅ Readable transcript saved to: {txt_path}")
        
        # Save SenseVoice output
        sensevoice_path = os.path.join(output_dir, f"{filename_base}_sensevoice.txt")
        with open(sensevoice_path, "w", encoding="utf-8") as f:
            f.write(enhanced_result['sensevoice_tagged'])
        print(f"✅ SenseVoice output saved to: {sensevoice_path}")

    def write_transcription_to_json(self, result, output_dir, filename_base):
        """Write the full transcription result to a JSON file."""
        output_path = os.path.join(output_dir, f"{filename_base}.json")
        
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
            
        print(f"✅ JSON transcript saved to: {output_path}")

In [6]:
def synthesize_speech(transcript, reference_audio_path, output_dir="../output"):
    """Synthesize speech using Fish Speech for each segment in the transcript.
    
    Args:
        transcript: Transcript JSON containing segments of text with speaker information
        reference_audio_path: Path to reference audio file
        output_dir: Directory to save the output files
        
    Returns:
        List of paths to the generated audio files
    """
    print("Synthesizing speech with Fish Speech...")
    try:
        client = Client("http://fish-speech:7860/")
        generated_files = []
        
        # Iterate through each segment in the transcript
        for segment in transcript["result"]["segments"]:
            text = segment["text"]
            speaker = segment["speaker"]
            
            # Synthesize speech for the segment
            result = client.predict(
                text=text,
                reference_id="",
                reference_audio=handle_file(reference_audio_path),
                reference_text="",
                max_new_tokens=0,
                chunk_length=200,
                top_p=0.5,
                repetition_penalty=1.7,
                temperature=0.6,
                seed=0,
                use_memory_cache="on",
                api_name="/partial"
            )
            
            # Handle the result
            if isinstance(result, tuple) and len(result) > 0:
                if isinstance(result[0], str) and result[0].endswith(".wav"):
                    wav_file_path = result[0]
                    
                    # Create output directory if it doesn't exist
                    os.makedirs(output_dir, exist_ok=True)
                    
                    # Save the file with a unique name based on the speaker and segment
                    output_file_path = os.path.join(output_dir, f"{speaker}_{os.path.basename(wav_file_path)}")
                    shutil.move(wav_file_path, output_file_path)
                    print(f"Generated .wav file for {speaker} saved to {output_file_path}")
                    generated_files.append(output_file_path)
                else:
                    print(f"Unexpected result format for segment: {result}")
            else:
                print(f"Unexpected result format for segment: {result}")
        
        return generated_files
    except Exception as e:
        print(f"Error in speech synthesis: {e}")
        return None

In [7]:
def process_audio_file(audio_file_path, reference_audio_path=None, whisper_model_size="base", language=None, output_dir="../output", model_id="MaziyarPanahi/Meta-Llama-3-8B-Instruct-GPTQ"):
    """Process an audio file: enhanced transcribe with emotions and then synthesize.
    
    Args:
        audio_file_path: Path to the audio file to transcribe
        reference_audio_path: Path to reference audio for synthesis (defaults to input audio if None)
        whisper_model_size: Size of the Whisper model to use
        language: Optional language code for transcription
        output_dir: Directory to save output files
        model_id: Model ID for translation
    
    Returns:
        Tuple of (path to generated audio file, path to transcription file, transcribed text, enhanced_result)
    """
    print(f"🎯 Processing audio file: {audio_file_path}")
    
    # Initialize the enhanced transcriber
    try:
        transcriber = AudioTranscriber(
            whisper_model_size=whisper_model_size,
            sensevoice_model_dir="iic/SenseVoiceSmall"
        )
    except Exception as e:
        print(f"❌ Failed to initialize enhanced transcriber: {e}")
        raise
    
    # Use the same audio file as reference if none provided
    if reference_audio_path is None:
        reference_audio_path = audio_file_path
    
    # Get base filename without extension for consistent naming
    base_filename = os.path.splitext(os.path.basename(audio_file_path))[0]
    
    # Create enhanced transcription subdirectory
    enhanced_output_dir = os.path.join(output_dir, "enhanced_transcription")
    
    # Updated usage example with diarization options
    video_path = "../input/job.mp4"
    pipeline_paths = full_video_audio_pipeline(
        video_path, 
        enable_diarization=True,  # Enable speaker separation
        min_speakers=1,           # Minimum number of speakers
        max_speakers=2,           # Maximum number of speakers  
        language=""             # Language (optional, auto-detect if None)
    )


# Optional: Enhanced synthesis function that can use emotion information
def synthesize_speech_with_emotions(enhanced_result, reference_audio_path, output_dir="../output"):
    """
    Enhanced speech synthesis that could potentially use emotion information.
    For now, uses the clean text, but structure allows for future emotion-aware synthesis.
    """
    print("🎤 Synthesizing speech with emotion awareness...")
    
    # Extract clean text
    clean_text = enhanced_result['text']
    
    # Future enhancement: Could modify synthesis based on detected emotions
    # For now, use standard synthesis
    output_audio_path = synthesize_speech(
        text=clean_text,
        reference_audio_path=reference_audio_path,
        output_dir=output_dir
    )
    
    # Could also create emotion-annotated audio segments in the future
    return output_audio_path


In [ ]:
def main():
    print("="*60)
    print("STARTING ENHANCED AUDIO PROCESSING PIPELINE")
    print("="*60)
    video_path = "../input/job.mp4"
    paths = generate_pipeline_paths(video_path, base_output)
    # Step 1: Separate vocals and no-vocals
    print("\n🎵 Step 1: Separating vocals from background...")
    vocals_path, no_vocals_path = separate_vocals(paths["audio_raw"],paths["root"])
    print(f"✅ Vocals extracted to: {vocals_path}")

    # Step 2: Convert vocals to 48kHz WAV
    print("\n🔄 Step 2: Converting to 48kHz WAV...")
    convert_to_48k_wav(vocals_path, paths["audio_vocals_48k"])




    # Step 3: Denoise the converted WAV
    print("\n🎚️  Step 3: Denoising audio...")
    denoise_audio(paths["audio_vocals_48k"], paths[ "audio_vocals_denoised"])

    
    # Step 4: Enhanced transcription and synthesis pipeline
    print("\n🎭 Step 4: Running enhanced transcription and synthesis pipeline...")
    try:
        # Use the best quality audio available
        best_audio = paths[ "audio_vocals_denoised"] if os.path.exists(paths[ "audio_vocals_denoised"]) else converted_wav
        if not os.path.exists(best_audio):
            best_audio = input_audio
            
        output_audio_path, text_file_path, transcribed_text, enhanced_result = process_audio_file(
            audio_file_path=best_audio,
            reference_audio_path=paths["audio_raw"],
            whisper_model_size="base",
            language="",  # Set to the language of the audio or None for auto-detection
            output_dir=paths["root"]
        )
        
        if output_audio_path and text_file_path:
            print(f"✅ Enhanced pipeline completed successfully!")
        else:
            print("❌ Pipeline failed to complete successfully.")
            return
            
    except Exception as e:
        print(f"❌ Error in enhanced pipeline: {e}")
        return
    
    # Step 5: Generate summary report
    print("\n📋 PIPELINE COMPLETION SUMMARY")
    print("="*60)
    
    # Display emotion highlights
    emotion_segments = [s for s in enhanced_result['segments'] if s['emotions']]
    if emotion_segments:
        print(f"\n🌟 Emotional highlights found:")
        for i, segment in enumerate(emotion_segments[:5]):
            timestamp = f"[{segment['start']:.1f}s-{segment['end']:.1f}s]"
            emotions = f"[{', '.join(segment['emotions'])}]"
            preview = segment['text'][:50] + "..." if len(segment['text']) > 50 else segment['text']
            print(f"   {i+1}. {timestamp} {emotions}: {preview}")
    
    print(f"\n🎉 Enhanced pipeline processing completed!")
    print(f"🎭 Detected emotions: {', '.join(set(enhanced_result['detected_emotions']))}")

if __name__ == "__main__":
    main()


STARTING ENHANCED AUDIO PROCESSING PIPELINE

🎵 Step 1: Separating vocals from background...
Separating vocals from ../output/job/audio/original_unprocessed.wav...
Selected model is a bag of 4 models. You will see that many progress bars per track.
Separated tracks will be stored in /DubbAI/notebooks/separated/mdx_extra
Separating track ../output/job/audio/original_unprocessed.wav


 96%|████████████████████████████████████████████████████████████████████▏  | 792.0/825.0 [00:08<00:00, 100.73seconds/s]/venv/lib/python3.10/site-packages/torch/nn/modules/conv.py:306: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv1d(input, weight, bias, self.stride,
100%|████████████████████████████████████████████████████████████████████████| 825.0/825.0 [00:08<00:00, 97.46seconds/s]
100%|████████████████████████████████████████████████████████████████████████| 825.0/825.0 [00:08<00:00, 96.52seconds/s]
 96%|█████████████████████████████████████████████████████████████████████   | 792.0/825.0 [00:08<00:00, 97.24seconds/s]/venv/lib/python3.10/site-packages/torch/nn/modules/conv.py:456: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinaliz

Vocals saved to: ../output/job/original_unprocessed_vocals.wav
No vocals saved to: ../output/job/original_unprocessed_no_vocals.wav
✅ Vocals extracted to: ../output/job/original_unprocessed_vocals.wav

🔄 Step 2: Converting to 48kHz WAV...
Converting ../output/job/original_unprocessed_vocals.wav to 48kHz WAV at ../output/job/audio/vocals_48k.wav...


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Conversion done.

🎚️  Step 3: Denoising audio...
Denoising audio ../output/job/audio/vocals_48k.wav to ../output/job/audio/vocals_denoised.wav...
2025-06-30 10:46:44 | INFO     | DF | Loading model settings of DeepFilterNet3
2025-06-30 10:46:44 | INFO     | DF | Using DeepFilterNet3 model at /root/.cache/DeepFilterNet/DeepFilterNet3
2025-06-30 10:46:44 | INFO     | DF | Initializing model `deepfilternet3`
2025-06-30 10:46:44 | INFO     | DF | Found checkpoint /root/.cache/DeepFilterNet/DeepFilterNet3/checkpoints/model_120.ckpt.best with epoch 120
2025-06-30 10:46:44 | INFO     | DF | Running on device cuda:0
2025-06-30 10:46:44 | INFO     | DF | Model loaded
